# 2. Analyze Results

This notebook loads local outputs, emits the consolidated summary tables, and plots context-length comparisons plus AttnRes depth usage.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

def find_repo_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent, Path('/content/AttnResGPT-next-scale')]
    for candidate in candidates:
        if (candidate / 'src' / 'training' / 'train.py').exists() and (candidate / 'requirements.txt').exists():
            return candidate.resolve()
    raise FileNotFoundError(
        'Could not find AttnResGPT-next-scale. Open the notebook from the repo root or sync the repo to /content/AttnResGPT-next-scale.'
    )

REPO_DIR = find_repo_root()
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print(f'Using repo at: {REPO_DIR}')


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid')
summary_df = pd.read_csv('outputs/logs/run_summaries.csv')
consolidated_df = pd.read_csv('outputs/logs/consolidated_summary_table.csv')
paired_df = pd.read_csv('outputs/logs/paired_comparisons.csv')
display(consolidated_df.sort_values(['size', 'context', 'model']))
display(paired_df.sort_values(['size', 'context']))
if Path('outputs/summary_large.csv').exists():
    display(pd.read_csv('outputs/summary_large.csv').sort_values(['context', 'model']))
if Path('outputs/summary_large_comparison.csv').exists():
    display(pd.read_csv('outputs/summary_large_comparison.csv').sort_values(['context']))


In [ ]:
plot_dir = Path('outputs/plots')
plot_dir.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.lineplot(data=summary_df, x='context', y='val_loss', hue='model', style='size', marker='o', ax=axes[0])
axes[0].set_title('Validation Loss vs Context')
sns.lineplot(data=summary_df, x='context', y='perplexity', hue='model', style='size', marker='o', ax=axes[1])
axes[1].set_title('Perplexity vs Context')
fig.tight_layout()
fig.savefig(plot_dir / 'loss_and_perplexity_vs_context.png', dpi=200)
plt.show()


In [ ]:
attnres_rows = summary_df[summary_df['model'] == 'attnres'][['run_name', 'context']].sort_values('context')
for _, row in attnres_rows.iterrows():
    summary_path = Path('outputs/runs') / row['run_name'] / 'run_summary.json'
    payload = json.loads(summary_path.read_text(encoding='utf-8'))
    heatmap = payload.get('depth_attention_rows', [])
    if not heatmap:
        continue
    plt.figure(figsize=(8, 4))
    sns.heatmap(heatmap, cmap='viridis')
    plt.title(f"AttnRes depth attention: {row['run_name']}")
    plt.xlabel('Source index')
    plt.ylabel('Depth-mixing row')
    plt.tight_layout()
    output_path = plot_dir / f"depth_heatmap_{row['run_name']}.png"
    plt.savefig(output_path, dpi=200)
    plt.show()
